In [1]:
import pandas as pd

In [2]:
current_resuts = 'output/LUVit_ct_by_tod_generator-2026-03-19_90-90-90.xlsx'
original_results = 'J:/Projects/LandUseVision/LUV.3/Control_totals/SubJuris_ConTots/LUVit_ct_by_tod_generator_90-90-90_2024-10-18_6575_LUVit_Match.xlsx'

In [3]:
df_current = pd.read_excel(current_resuts, sheet_name='unrolled')
df_original = pd.read_excel(original_results, sheet_name='unrolled')

In [4]:
key_cols = ['subreg_id', 'year']
value_cols = ['total_hhpop', 'total_hh', 'total_emp']

print('Shape check')
print('  current :', df_current.shape)
print('  original:', df_original.shape)
print('  row count match:', len(df_current) == len(df_original))
print()

print('Column check')
print('  current columns :', list(df_current.columns))
print('  original columns:', list(df_original.columns))
print('  columns match:', list(df_current.columns) == list(df_original.columns))
print()

print('Key integrity check')
print('  current duplicate keys :', df_current.duplicated(key_cols).sum())
print('  original duplicate keys:', df_original.duplicated(key_cols).sum())
print()

merged = df_current.merge(
    df_original,
    on=key_cols,
    how='outer',
    suffixes=('_current', '_original'),
    indicator=True,
)

print('Coverage check')
print(merged['_merge'].value_counts(dropna=False).rename_axis('status').to_frame('rows'))
print()

matched = merged.loc[merged['_merge'] == 'both'].copy()
for col in value_cols:
    matched[f'{col}_diff'] = matched[f'{col}_current'] - matched[f'{col}_original']

summary = pd.DataFrame(
    {
        'exact_match_rows': [
            (matched[f'{col}_current'] == matched[f'{col}_original']).sum() for col in value_cols
        ],
        'mismatch_rows': [
            (matched[f'{col}_current'] != matched[f'{col}_original']).sum() for col in value_cols
        ],
        'max_abs_diff': [matched[f'{col}_diff'].abs().max() for col in value_cols],
        'sum_current': [matched[f'{col}_current'].sum() for col in value_cols],
        'sum_original': [matched[f'{col}_original'].sum() for col in value_cols],
        'sum_diff': [matched[f'{col}_diff'].sum() for col in value_cols],
    },
    index=value_cols,
)

print('Value comparison summary')
display(summary)

mismatch_mask = False
for col in value_cols:
    mismatch_mask = mismatch_mask | (matched[f'{col}_current'] != matched[f'{col}_original'])

mismatches = matched.loc[
    mismatch_mask,
    key_cols
    + [f'{col}_current' for col in value_cols]
    + [f'{col}_original' for col in value_cols]
    + [f'{col}_diff' for col in value_cols],
].sort_values(key_cols)

print('Sample mismatched rows')
display(mismatches.head(20))
print('Total mismatched rows:', len(mismatches))
print()

current_year_totals = df_current.groupby('year')[value_cols].sum().sort_index()
original_year_totals = df_original.groupby('year')[value_cols].sum().sort_index()
year_diff = (
    current_year_totals
    .rename(columns=lambda col: f'{col}_current')
    .join(original_year_totals.rename(columns=lambda col: f'{col}_original'))
)
for col in value_cols:
    year_diff[f'{col}_diff'] = year_diff[f'{col}_current'] - year_diff[f'{col}_original']

print('Year-level totals comparison')
display(year_diff)

Shape check
  current : (1477, 5)
  original: (1477, 5)
  row count match: True

Column check
  current columns : ['subreg_id', 'year', 'total_hhpop', 'total_hh', 'total_emp']
  original columns: ['subreg_id', 'year', 'total_hhpop', 'total_hh', 'total_emp']
  columns match: True

Key integrity check
  current duplicate keys : 0
  original duplicate keys: 0

Coverage check
            rows
status          
both        1477
left_only      0
right_only     0

Value comparison summary


,exact_match_rows,mismatch_rows,max_abs_diff,sum_current,sum_original,sum_diff
total_hhpop,16,1461,29740,34300998,34913064,-612066
total_hh,32,1445,24134,13743304,14289457,-546153
total_emp,24,1453,16144,19481874,19860606,-378732


Sample mismatched rows


,subreg_id,year,total_hhpop_current,total_hh_current,total_emp_current,total_hhpop_original,total_hh_original,total_emp_original,total_hhpop_diff,total_hh_diff,total_emp_diff
0,1,2020,82397,30833,42265,87781,32391,42405,-5384,-1558,-140
1,1,2025,84249,31499,42912,89632,33054,43976,-5383,-1555,-1064
2,1,2030,86101,32164,43558,91483,33717,45546,-5382,-1553,-1988
3,1,2035,87954,32830,44204,93334,34380,47117,-5380,-1550,-2913
4,1,2040,89806,33496,44851,95184,35044,48688,-5378,-1548,-3837
5,1,2044,91287,34028,45368,96665,35574,49944,-5378,-1546,-4576
6,1,2050,93510,34827,46144,98886,36370,51829,-5376,-1543,-5685
7,2,2020,208484,88026,109348,209994,90308,112371,-1510,-2282,-3023
8,2,2025,214548,90024,112611,215363,92634,115626,-815,-2610,-3015
9,2,2030,220612,92022,115875,220731,94959,118881,-119,-2937,-3006


Total mismatched rows: 1477

Year-level totals comparison


,total_hhpop_current,total_hh_current,total_emp_current,total_hhpop_original,total_hh_original,total_emp_original,total_hhpop_diff,total_hh_diff,total_emp_diff
year,,,,,,,,,
2020,4053154,1605263,2232229,4209191,1670236,2312316,-156037,-64973,-80087
2025,4338191,1725768,2417628,4471148,1795129,2488970,-132957,-69361,-71342
2030,4623243,1846265,2603021,4733113,1920025,2665623,-109870,-73760,-62602
2035,4908291,1966773,2788421,4995070,2044925,2842275,-86779,-78152,-53854
2040,5193331,2087279,2973822,5257018,2169814,3018930,-63687,-82535,-45108
2044,5421368,2183675,3122139,5466584,2269725,3160255,-45216,-86050,-38116
2050,5763420,2328281,3344614,5780940,2419603,3372237,-17520,-91322,-27623


In [ ]:
r_script_inputs = 'data/control_id_working.xlsx'
original_r_script_inputs = 'J:/Projects/LandUseVision/LUV.3/Control_totals/control_id_working_111522.xlsx'

df_current_inputs = pd.read_excel(r_script_inputs)
df_original_inputs = pd.read_excel(original_r_script_inputs)

In [6]:
key_col_inputs = 'control_id'
value_cols_inputs = [
    c for c in df_current_inputs.columns
    if c != key_col_inputs and pd.api.types.is_numeric_dtype(df_current_inputs[c])
]

print('Shape check')
print('  current :', df_current_inputs.shape)
print('  original:', df_original_inputs.shape)
print('  row count match:', len(df_current_inputs) == len(df_original_inputs))
print()

print('Column check')
current_cols  = list(df_current_inputs.columns)
original_cols = list(df_original_inputs.columns)
print('  current columns :', current_cols)
print('  original columns:', original_cols)
print('  columns match:', current_cols == original_cols)
extra_current  = [c for c in current_cols  if c not in original_cols]
extra_original = [c for c in original_cols if c not in current_cols]
if extra_current:
    print('  columns only in current :', extra_current)
if extra_original:
    print('  columns only in original:', extra_original)
print()

print('Key integrity check')
print('  current duplicate keys :', df_current_inputs.duplicated(key_col_inputs).sum())
print('  original duplicate keys:', df_original_inputs.duplicated(key_col_inputs).sum())
print()

merged_inputs = df_current_inputs.merge(
    df_original_inputs,
    on=key_col_inputs,
    how='outer',
    suffixes=('_current', '_original'),
    indicator=True,
)

print('Coverage check')
print(merged_inputs['_merge'].value_counts(dropna=False).rename_axis('status').to_frame('rows'))
print()

shared_value_cols = [
    c for c in value_cols_inputs
    if c in df_original_inputs.columns
]

matched_inputs = merged_inputs.loc[merged_inputs['_merge'] == 'both'].copy()
for col in shared_value_cols:
    matched_inputs[f'{col}_diff'] = matched_inputs[f'{col}_current'] - matched_inputs[f'{col}_original']

summary_inputs = pd.DataFrame(
    {
        'exact_match_rows': [
            (matched_inputs[f'{col}_current'] == matched_inputs[f'{col}_original']).sum()
            for col in shared_value_cols
        ],
        'mismatch_rows': [
            (matched_inputs[f'{col}_current'] != matched_inputs[f'{col}_original']).sum()
            for col in shared_value_cols
        ],
        'max_abs_diff': [matched_inputs[f'{col}_diff'].abs().max() for col in shared_value_cols],
        'sum_current':  [matched_inputs[f'{col}_current'].sum()  for col in shared_value_cols],
        'sum_original': [matched_inputs[f'{col}_original'].sum() for col in shared_value_cols],
        'sum_diff':     [matched_inputs[f'{col}_diff'].sum()     for col in shared_value_cols],
    },
    index=shared_value_cols,
)

print('Value comparison summary')
display(summary_inputs)

mismatch_mask_inputs = False
for col in shared_value_cols:
    mismatch_mask_inputs = mismatch_mask_inputs | (
        matched_inputs[f'{col}_current'] != matched_inputs[f'{col}_original']
    )

mismatches_inputs = matched_inputs.loc[
    mismatch_mask_inputs,
    [key_col_inputs]
    + [f'{col}_current'  for col in shared_value_cols]
    + [f'{col}_original' for col in shared_value_cols]
    + [f'{col}_diff'     for col in shared_value_cols],
].sort_values(key_col_inputs)

print('Sample mismatched rows')
display(mismatches_inputs.head(20))
print('Total mismatched rows:', len(mismatches_inputs))


Shape check
  current : (155, 36)
  original: (155, 28)
  row count match: True

Column check
  current columns : ['target_id', 'name', 'control_id', 'county_id', 'exclude_from_target', 'RGID', 'TotPop20', 'HHpop20', 'Units20', 'HH20', 'GQ20', 'TotEmp20_wCRnoMil', 'Emp_ConRes', 'TotEmpNoMil-ResCon', 'Emp_Military', 'Emp_Total', 'hh_2044', 'total_pop_2044', 'gq_2044', 'hhpop_2044', 'emp_2044', 'hh_2050', 'TotPop50', 'gq_2050', 'hhpop_2050', 'TotEmp50_wCRnoMil', 'Pop18', 'HHpop18', 'Units18', 'HH18', 'GQ18', 'Emp18', 'TotEmpTrg_wCRnoMil', 'TotPopTrg', 'GQpct50', 'PPH50']
  original columns: ['control_id', 'name', 'county_id', 'RGID', 'Pop18', 'HHpop18', 'HH18', 'Emp18', 'TotPop20', 'HHpop20', 'GQpct20', 'HH20', 'PPH20', 'TotPopTrg', 'TotPop44', 'HHPop44', 'HH44', 'TotPop50', 'HHpop50', 'GQpct50', 'HH50', 'PPH50', 'TotEmp20_wCRnoMil', 'Mil20', 'CR20', 'TotEmpTrg_wCRnoMil', 'TotEmp44_wCRnoMil', 'TotEmp50_wCRnoMil']
  columns match: False
  columns only in current : ['target_id', 'exclude_f

,exact_match_rows,mismatch_rows,max_abs_diff,sum_current,sum_original,sum_diff
county_id,155,0,0.000000,7.047000e+03,7.047000e+03,0.000000
RGID,148,7,2.000000,5.770000e+02,5.850000e+02,-8.000000
TotPop20,10,145,22622.050000,4.240977e+06,4.294373e+06,-53395.897326
HHpop20,10,145,15228.050000,4.174340e+06,4.209191e+06,-34851.006850
HH20,6,149,4329.925081,1.658217e+06,1.670236e+06,-12019.187445
TotEmp20_wCRnoMil,155,0,0.000000,2.312316e+06,2.312316e+06,0.000000
TotPop50,18,137,29720.254864,5.885673e+06,5.884892e+06,781.027252
TotEmp50_wCRnoMil,29,126,15660.500000,3.424699e+06,3.372240e+06,52458.856972
Pop18,1,154,24611.880664,4.106712e+06,4.133430e+06,-26718.300138
HHpop18,1,154,19746.288600,4.036416e+06,4.052115e+06,-15699.147427


Sample mismatched rows


,control_id,county_id_current,RGID_current,TotPop20_current,HHpop20_current,HH20_current,TotEmp20_wCRnoMil_current,TotPop50_current,TotEmp50_wCRnoMil_current,Pop18_current,...,TotPop50_diff,TotEmp50_wCRnoMil_diff,Pop18_diff,HHpop18_diff,HH18_diff,Emp18_diff,TotEmpTrg_wCRnoMil_diff,TotPopTrg_diff,GQpct50_diff,PPH50_diff
0,1,33,1,151150.863095,149710.863095,60642.312935,155023,249206.0,249639,144038.002057,...,1765.960872,371.849080,1719.002057,1475.645055,374.873511,0,19220.679264,21586.305603,-0.702846,0.038691
1,2,33,1,728607.472765,705004.972765,344496.110901,691359,1010712.0,887154,705991.119336,...,-6940.464187,488.806195,-24611.880664,-19746.288600,-4443.595330,0,39550.044956,57594.555885,-3.550099,0.153346
2,3,33,2,77160.205546,76244.205546,27024.662421,49459,109758.0,76203,74435.503495,...,1357.320004,1097.549845,3628.503495,3407.664500,707.858930,0,6226.839876,7671.650457,-1.020596,0.059540
3,4,33,2,28887.662318,28598.662318,11977.405698,17875,41794.0,30174,26784.370605,...,-356.589058,14.203763,-768.629395,-672.251401,-120.772050,0,2471.163010,2350.666436,-0.827931,0.042218
4,5,33,2,52023.526062,51505.526062,19855.839156,14064,71125.0,20473,51440.362113,...,-261.764223,106.690561,-867.637887,-898.403890,-164.805998,0,1367.152448,3644.862560,-0.876257,0.048043
5,6,33,2,100556.088586,99442.088586,35965.019998,32257,129679.0,58868,98688.838807,...,761.334828,167.828115,1487.838807,1256.201814,265.175989,0,5456.462492,6819.979276,-1.043606,0.059744
6,7,33,2,39019.789045,38491.789045,16025.578451,30610,45534.0,39491,36843.425097,...,-724.949889,4.216976,-245.574903,-392.360903,-353.724471,0,1779.573581,1547.851044,-1.378176,0.066775
7,8,33,2,135351.328864,133839.328864,46501.307403,78269,160354.0,124232,131435.113564,...,1816.214774,362.185901,2854.113564,3074.427548,-48.760669,0,9482.348721,7442.842955,-1.161780,0.067980
8,9,33,2,92068.425423,90923.425423,37998.401188,57161,122246.0,89395,88169.103653,...,689.064193,-47.833031,796.103653,671.003657,544.337432,0,6408.533575,6672.025931,-1.137592,0.056575
9,10,33,2,73011.026711,72589.026711,29590.344561,101631,119684.0,129355,65793.772634,...,1788.598317,56.065949,2117.772634,2042.690626,679.053280,0,5589.652760,10961.451943,-0.432333,0.024478


Total mismatched rows: 155


In [8]:
mismatches_inputs.sort_values('HH20_diff')[['control_id','HH20_current','HH20_original','HH20_diff']]

,control_id,HH20_current,HH20_original,HH20_diff
153,403,182.074919,4512,-4329.925081
150,304,4065.264841,6771,-2705.735159
149,303,396.964144,2664,-2267.035856
147,301,2851.726559,4703,-1851.273441
1,2,344496.110901,345627,-1130.889099
...,...,...,...,...
49,53,633.555934,528,105.555934
65,76,38475.681673,38242,233.681673
54,64,43019.539914,41306,1713.539914
91,124,61251.109434,59103,2148.109434
